# Description

Gemma is a family of lightweight, state-of-the-art open models from Google, built from the same research and technology used to create the Gemini models. They are text-to-text, decoder-only large language models, available in English, with open weights, pre-trained variants, and instruction-tuned variants. Gemma models are well-suited for a variety of text generation tasks, including question answering, summarization, and reasoning. Their relatively small size makes it possible to deploy them in environments with limited resources such as a laptop, desktop or your own cloud infrastructure, democratizing access to state of the art AI models and helping foster innovation for everyone..

# Setup

### Install Dependencies
Install keras, kerasNLP and other dependencies

In [1]:
# Install Keras 3 last. See https://keras.io/getting_started/ for more details.
%pip install -q -U keras-nlp
%pip install -q -U keras>=3

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Select a backend

Keras is a high-level, multi-framework deep learning API designed for simplicity and ease of use. Using Keras 3, you can run workflows on one of three backends: TensorFlow, JAX, or PyTorch.

In [2]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "tensorflow" or "torch".

**Import packages**

Import Keras and KerasNLP.

In [3]:
# %pip install -q jax jaxlib

import keras
import keras_nlp

# **Gemma: Fine-Tuning Large Language Models with Low Rank Adaptation (LoRA)**

**Low Rank Adaptation (LoRA)** is a fine-tuning technique for Large Language Models (LLMs) like Gemma. It reduces the number of trainable parameters by freezing existing model weights and introducing a smaller set of new weights, enhancing training speed, memory efficiency, and resulting in more compact model weights while maintaining high-quality outputs.

#### **Key Questions Addressed:**
* **Why Fine-Tune LLMs?** Fine-tuning allows customization for specific tasks, adapting pre-trained models to domain-specific nuances.

* **What is Used for Fine-Tuning LLMs?** The tutorial utilizes Low Rank Adaptation (LoRA), a technique that optimizes the fine-tuning process for Gemma models, balancing efficiency and model quality.

# Preparing Dataset

In [4]:
import pandas as pd

df_questions = pd.read_csv('../data/PythonQuestionsfromStackOverflow/Questions.csv',
                            encoding = "ISO-8859-1",
                            usecols = ['Id','Score','Title'])
#answers table
df_answers = pd.read_csv('../data/PythonQuestionsfromStackOverflow/Answers.csv',
                            encoding = "ISO-8859-1",
                            usecols = ['ParentId','Score','Body'],#parent id links to the questions table
                            )

df_questions = df_questions[df_questions['Score'] > 0]
df_answers = df_answers[df_answers['Score'] > 0]\
    .sort_values('Score',ascending=False)\
    .drop_duplicates(subset=['ParentId'])

In [5]:
qa = df_questions.merge(df_answers,left_on = 'Id', right_on = 'ParentId')\
    .rename(columns={'Title':'Question','Body':'Answer'})[['Question','Answer','Score_x']]

data = []
for index, row in qa.iterrows():
    data.append(f"Question:\n{row['Question']}\n\nAnswer:\n{row['Answer']}")
    
# DataSet Created

data = data[:500]
# Preprocess the data.we uses a subset of 500 training examples to execute the notebook faster. Consider using more training data for higher quality fine-tuning.

# Load Model

In [ ]:
import os
import keras
import keras_nlp
from huggingface_hub import login
import getpass

# Install required dependencies for Gemma tokenizer
# %pip install -q tensorflow tensorflow-text

# Set up environmental variables for model download
os.environ["GEMMA_MODEL_CACHE_DIR"] = "./models/"  # Optional: specify model cache directory

# You need to have a Hugging Face account and accept the Gemma model terms
# Make sure you've accepted the model terms at https://huggingface.co/google/gemma-2b
# and have proper authentication

from dotenv import load_dotenv
load_dotenv()

# Authenticate with Hugging Face
hf_token = os.getenv("HuggingFace")
if not hf_token:
	print("Please enter your Hugging Face token to download the Gemma model.")
	print("If you don't have one, create an account at https://huggingface.co/")
	print("Then get your token at https://huggingface.co/settings/tokens")
	print("You also need to accept the model terms at https://huggingface.co/google/gemma-2b")
	hf_token = getpass.getpass("Enter your Hugging Face token: ")

login(token=hf_token)

# Load the pre-trained Gemma model
# When using a Hugging Face model, we need to prefix with "hf://"
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("hf://google/gemma-2b")
# from_preset instantiates the model from a preset architecture and weights
# In the code above, we're using the official model ID from Hugging Face with proper format
gemma_lm.summary()

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

e:\OneDrive\repo\ai_ml_experiments\aiml1\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Jamie\.cache\huggingface\hub\models--google--gemma-2b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

ImportError: GemmaTokenizer requires `tensorflow` and `tensorflow-text` for text processing. Run `pip install tensorflow-text` to install both packages or visit https://www.tensorflow.org/install

If `tensorflow-text` is already installed, try importing it in a clean python session. Your installation may have errors.

KerasHub uses `tf.data` and `tensorflow-text` to preprocess text on all Keras backends. If you are running on Jax or Torch, this installation does not need GPU support.

# **LoRA Fine-tuning**

**Low Rank Adaptation (LoRA)** is a fine-tuning technique for Large Language Models (LLMs) like Gemma. It reduces the number of trainable parameters by freezing existing model weights and introducing a smaller set of new weights, enhancing training speed, memory efficiency, and resulting in more compact model weights while maintaining high-quality outputs.

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Tokenizer (type)                                   ┃                                             Vocab # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                   │                                             256,000 │
└────────────────────────────────────────────────────┴─────────────────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,507,536,384 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,507,536,384 (9.34 GB)

 Trainable params: 1,363,968 (5.20 MB)

 Non-trainable params: 2,506,172,416 (9.34 GB)

In [ ]:
# Limit the input sequence length to 128 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 128
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(data, epochs=1, batch_size=1)

500/500 ━━━━━━━━━━━━━━━━━━━━ 131s 223ms/step - loss: 1.8924 - sparse_categorical_accuracy: 0.5868


In [ ]:
def ask_question(query:str)->str:
    template = "Question:\n{question}\n\nAnswer:\n{answer}"
    prompt = template.format(
        question=query,
        answer="",
    )
    return prompt

sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)

In [ ]:
prompt = ask_question("What is Python?")
print(gemma_lm.generate(prompt, max_length=512))

Question:
What is Python?

Answer:
<p>Python is an interpreted, interactive, object-oriented programming language, widely used for web development and scripting. It is a dynamically-typed programming language which means that the type of data is determined at runtime.</p>

<p>The language itself is simple and easy to learn, and the syntax closely follows English. This makes Python a great language for learning object-oriented design. It can be easily integrated with the standard libraries and the standard Python modules.</p>

<p>Python is an open source project, developed and maintained by the Python Software Foundation, with the Python core developers working at a non-profit foundation, the non-profit non-profit, the <a href="http://www.python.org/">Python Software Foundation</a>. The Python Software Foundation is a <a href="http://www.python.org/psf/membership.html">registered not-for-profit corporation</a> based in the United Kingdom with offices in San Francisco and London. The Pyt

In [ ]:
prompt = ask_question("How to implement a stack in Python?")
print(gemma_lm.generate(prompt, max_length=512))

Question:
How to implement a stack in Python?

Answer:
<blockquote><code>

def stack():
    """
    Create a Stack class.
    """
    return []


class Stack(object):
    """
    A Stack class.
    """

    def __init__(self):
        self._items = stack()


    def is_empty(self):
        """
        Return True if this stack has no items.
        """

        return self._items == stack()


    def push(self, item):
        """
        Add `item` to the top of this stack.
        """

        self._items.append(item)


    def pop(self):
        """
        Remove and return the item at the top of this stack, or raise a
        ValueError if this stack is empty.
        """

        if self._items != stack():
            return self._items.pop()


    def top(self):
        """
        Return (without removing) the item at the top of this stack, or
        raise a ValueError if this stack is empty.
        """

        if self._items != stack():
            return self._items.top()


# Resources
* https://ai.google.dev/gemma/docs
* https://ai.google.dev/gemma/docs/lora_tuning
* Python Questions dataset: https://www.kaggle.com/datasets/stackoverflow/pythonquestions 